### Imports

In [1]:
import numpy as np
from faker import Faker
from sentence_transformers import SentenceTransformer

import pymc as pm
import pandas as pd
import random, re

from services.privacy_check_service.privacy_checker import PrivacyChecker

### Load the embedding model

In [2]:
encoder_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Generate 100 names with the faker library

In [3]:
faker = Faker(locale="de_DE")

names = [faker.name() for _ in range(100)]
names

['Univ.Prof. Bertold Schottin',
 'Ing. Ljiljana Renner B.Sc.',
 'Ing. Ursel Rosenow',
 'Hanno Koch',
 'Bianca Mühle',
 'Univ.Prof. Klaus Dieter Stroh',
 'Ralf Heintze-Hiller',
 'Anna-Maria Stey B.Sc.',
 'Maria-Luise Scheel',
 'Leon Herrmann',
 'Gesa Caspar',
 'Michel Mangold',
 'Muzaffer Müller',
 'Prof. Erika Krebs',
 'Dr. Tibor Rust MBA.',
 'Univ.Prof. Ullrich Ehlert B.Sc.',
 'Dittmar Schmiedt B.Eng.',
 'Daniel Gute',
 'Rosalie Eberth',
 'Jan Hiller',
 'Valeska Kruschwitz',
 'Calogero Söding',
 'Rosemarie Säuberlich B.A.',
 'Ann-Kathrin Weitzel-Buchholz',
 'Alf Losekann MBA.',
 'Albrecht Hänel',
 'Burghard Pechel',
 'Dr. Ann-Kathrin Holt B.Eng.',
 'Isa Eberhardt',
 'Theodoros Dussen van',
 'Timo Stoll',
 'Herma Fechner',
 'Univ.Prof. Anka Gute B.Eng.',
 'Dr. Emil Bonbach B.Eng.',
 'Hansgeorg Renner',
 'Amalia Trapp',
 'Dierk Freudenberger-Hornich',
 'Heinz-Günther Ullrich',
 'Adelbert Birnbaum-Rudolph',
 'Marcel Striebitz B.Eng.',
 'Conrad Kitzmann MBA.',
 'Henrik Kaul B.Eng.',
 'Han

### Encode the text and the names

In [44]:
def generate_sentences(names, templates, count=10, seed=None):
    if seed is not None:
        random.seed(seed)
    sentences = []
    for _ in range(count):
        tmpl = random.choice(templates)
        keys = sorted(set(re.findall(r"\{(name\d*)\}", tmpl)), key=lambda s: (int(s[4:]) if s!="name" and s[4:].isdigit() else 0, s))
        chosen = random.sample(names, k=max(1, len(keys)))
        mapping = {k: chosen[i] for i, k in enumerate(keys)} if keys else {"name": random.choice(names)}
        sentences.append(tmpl.format(**mapping))
    return sentences

In [46]:
templates = [
    "{name} went to the market and found a rare book.",
    "{name} and {name2} argued about the best pizza topping.",
    "Yesterday, {name} solved a tricky bug in record time.",
    "Is {name} joining the meeting later?",
    "{name} sent {name2} a postcard from Berlin.",
    "{name} is responsible for the Zurich office."
]

In [50]:
sentences_names = list()

for s in generate_sentences(names, templates, count=100, seed=42):
    sentences_names.append(s)

In [59]:
sentences_names

['Jan Hörle MBA. is responsible for the Zurich office.',
 'Hans-Willi Geißler-Schüler went to the market and found a rare book.',
 'Yesterday, Univ.Prof. Irmtrud Trupp B.Eng. solved a tricky bug in record time.',
 'Walburga Sontag and Hans-Willi Geißler-Schüler argued about the best pizza topping.',
 'Felicitas Wagenknecht went to the market and found a rare book.',
 'Matthäus Matthäi-Hamann is responsible for the Zurich office.',
 'Evamaria Jessel went to the market and found a rare book.',
 'Is Dominik Karz joining the meeting later?',
 'Margot Gunpf went to the market and found a rare book.',
 'Arzu Gehringer and Prof. Kreszentia Hoffmann MBA. argued about the best pizza topping.',
 'Oswald Beer-Hethur sent Polina Ackermann a postcard from Berlin.',
 'Gunther Kallert and Adalbert Martin MBA. argued about the best pizza topping.',
 'Matthäus Matthäi-Hamann is responsible for the Zurich office.',
 'Is Herr Frithjof Weihmann joining the meeting later?',
 'Is Evamaria Jessel joining the

In [4]:
text = "Wally Finke is responsible for the Zurich office."
text_embeddings = encoder_model.encode(text)

name_embeddings = encoder_model.encode(names)

In [12]:
def levenshtein(a, b, ratio=True, print_matrix=False, lowercase=False) :
	if type(a) != type('') :
		raise TypeError('First argument is not a string!')
	if type(b) != type('') :
		raise TypeError('Second argument is not a string!')
	if a == '' :
		return len(b)
	if b == '' :
		return len(a)
	if lowercase :
		a = a.lower()
		b = b.lower()

	n = len(a)
	m = len(b)
	lev = np.zeros((n+1,m+1))

	for i in range(0,n+1) :
		lev[i,0] = i
	for i in range(0,m+1) :
		lev[0,i] = i

	for i in range(1,n+1) :
		for j in range(1,m+1) :
			insertion = lev[i-1,j] + 1
			deletion = lev[i,j-1] + 1
			substitution = lev[i-1,j-1] + (1 if a[i-1]!= b[j-1] else 0)
			lev[i,j] = min(insertion,deletion,substitution)

	if print_matrix :
		print(lev)

	if ratio :
		return (n+m-lev[n,m])/(n+m)
	else :
		return lev[n,m]

In [34]:
def make_features(query, candidate_name, embedding_similarity):
    return [
        embedding_similarity,

        # Exact match
        float(candidate_name.lower() in query.lower()),

        # Character similarity
        levenshtein(query, candidate_name),

        # z-score of the similarity
        (embedding_similarity - similarities.mean()) / similarities.std()
    ]

### Calculate the cosine similarity between the text and the names

In [30]:
similarities = name_embeddings @ text_embeddings
similarities

array([ 0.34372774,  0.20947625,  0.19558267,  0.0927439 ,  0.11160587,
        0.08192791,  0.06127395,  0.18603542,  0.12280534,  0.30424792,
        0.20630094,  0.20481823,  0.22137332,  0.26324472,  0.22526848,
        0.12289128,  0.21743082,  0.20614256,  0.21544701,  0.27104068,
        0.1995438 ,  0.26034868,  0.18535398,  0.21610737,  0.11603965,
       -0.02751007,  0.21362703,  0.19866756,  0.16204004,  0.24611083,
        0.1623455 ,  0.1400388 ,  0.11779494, -0.00102836,  0.09478138,
        0.23914286,  0.19322628,  0.18757045,  0.18911812,  0.25452548,
        0.16868258,  0.25070363,  0.15395854,  0.044398  ,  0.19716206,
        0.15288973, -0.0352222 , -0.01463695,  0.24262682,  0.16199689,
        0.18574706,  0.23254183,  0.17523661,  0.19604865,  0.12366535,
        0.16712058,  0.2469147 ,  0.2971271 ,  0.23794356,  0.31820577,
        0.2250784 ,  0.25805223,  0.08370385,  0.09602182,  0.23971236,
        0.17164655,  0.09760431,  0.11932813,  0.31175035,  0.08

### Similarity statistics

In [31]:
sims = np.array(similarities)

print("min:", sims.min())
print("max:", sims.max())
print("mean:", sims.mean())
print("std:", sims.std())

print(
    "top 10:",
    np.sort(sims)[-10:][::-1]
)

min: -0.035222203
max: 0.6810154
mean: 0.17836905
std: 0.09457668
top 10: [0.6810154  0.34372774 0.32547736 0.31820577 0.31780976 0.31308576
 0.31175035 0.30424792 0.2971271  0.27104068]


### Z-score of the similarity between the text and the name "Wally Finke"

In [32]:
z = (
    0.681015 - sims.mean()
) / sims.std()

print(z)

5.3146925


### Generate features for each name based on the text and the similarity

In [35]:
features_list = [make_features(text, name, sim) for name, sim in zip(names, similarities)]
features_list = np.array(features_list)

features_list.shape

(100, 4)

In [36]:
features_list

array([[ 3.43727738e-01,  0.00000000e+00,  4.22535211e-01,
         1.74840879e+00],
       [ 2.09476247e-01,  0.00000000e+00,  3.16666667e-01,
         3.28909844e-01],
       [ 1.95582673e-01,  0.00000000e+00,  5.37500000e-01,
         1.82007104e-01],
       [ 9.27439034e-02,  0.00000000e+00,  4.20289855e-01,
        -9.05351520e-01],
       [ 1.11605868e-01,  0.00000000e+00,  4.14285714e-01,
        -7.05915868e-01],
       [ 8.19279104e-02,  0.00000000e+00,  5.25641026e-01,
        -1.01971364e+00],
       [ 6.12739511e-02,  0.00000000e+00,  3.38709677e-01,
        -1.23809695e+00],
       [ 1.86035424e-01,  0.00000000e+00,  3.70967742e-01,
         8.10599327e-02],
       [ 1.22805342e-01,  0.00000000e+00,  4.36619718e-01,
        -5.87498963e-01],
       [ 3.04247916e-01,  0.00000000e+00,  4.44444444e-01,
         1.33097160e+00],
       [ 2.06300944e-01,  0.00000000e+00,  4.81012658e-01,
         2.95336008e-01],
       [ 2.04818234e-01,  0.00000000e+00,  3.43750000e-01,
      

### Bayesian approach to determine if the text contains a name

In [43]:
true_candidate = 0

with pm.Model() as model:

    beta = pm.Normal(
        "beta",
        mu=0.5,
        sigma=2.0,
        shape=features_list.shape[1]
    )

    logits = features_list @ beta

    p = pm.Deterministic(
        "p",
        pm.math.softmax(logits)
    )

    y = pm.Categorical(
        "y",
        p=p,
        observed=true_candidate
    )

    trace = pm.sample(
        2000,
        tune=2000,
        target_accept=0.99,
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 1 seconds.


In [46]:
posterior_p = (
    trace.posterior["p"]
    .mean(dim=("chain", "draw"))
    .values
)

similarity_probabilities = list()
for name, prob in zip(names, posterior_p):
    similarity_probabilities.append({
        "name": name,
        "similarity": similarities[names.index(name)],
        "probability": prob
    })

pd.DataFrame(similarity_probabilities)

,name,similarity,probability
0,Dr. Karl-Otto Mitschke,0.343728,0.020112
1,Janus Bloch,0.209476,0.006806
2,Dipl.-Ing. Renate Hauffer B.Sc.,0.195583,0.007593
3,Christos Schuchhardt,0.092744,0.004805
4,Türkan Textor-Zorbach,0.111606,0.004946
...,...,...,...
95,Gert Jessel,0.205639,0.006589
96,Konstanze Liebelt,0.034788,0.004550
97,Dipl.-Ing. Sylvana Hänel,0.127100,0.005121
98,Univ.Prof. Adem Mälzer MBA.,0.166261,0.006102


### Get the max risk row

In [48]:
df_probabilities = pd.DataFrame(similarity_probabilities)
[df_probabilities[df_probabilities["probability"] == df_probabilities["probability"].max()]]

[           name  similarity  probability
 99  Wally Finke    0.681015     0.295641]

### Display the result of the bayesian model

In [42]:
import arviz as az

summary = az.summary(
    trace,
    var_names=["p"],
)
summary

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
p[0],0.0187,0.0131,0.0028,0.042,4230,4243,1.00,0.00018,0.00021
p[1],0.00639,0.00324,0.00061,0.011,6681,5414,1.00,3.9e-05,2.1e-05
p[2],0.008,0.00478,0.00062,0.016,6093,5480,1.00,5.9e-05,4e-05
p[3],0.0049,0.00444,0.00011,0.014,6458,5324,1.00,5.7e-05,4.3e-05
p[4],0.005,0.00402,0.00015,0.013,6527,5415,1.00,5.1e-05,3e-05
p[5],0.0057,0.0059,9.5e-05,0.017,6436,5468,1.00,7.8e-05,8.9e-05
p[6],0.0044,0.0048,5.8e-05,0.014,6307,5164,1.00,6.5e-05,6.7e-05
p[7],0.00604,0.00317,0.00046,0.0098,6785,5500,1.00,3.7e-05,1.7e-05
p[8],0.0053,0.00399,0.00018,0.012,6598,5496,1.00,4.9e-05,2.7e-05
p[9],0.0137,0.0078,0.0024,0.027,4562,4560,1.00,0.00011,8.4e-05


### Test the service implementation

In [5]:
privacy_checker = PrivacyChecker()

risk_result = privacy_checker.check_privacy_risk(text, names)
risk_result

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 1 seconds.


{'name': {5: 'Univ.Prof. Klaus Dieter Stroh'},
 'risk_probability': {5: 0.23399277972728155}}